In [1]:
#层和块
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))#三层

X = torch.rand(2,20)
net(X)

tensor([[ 0.0952,  0.1074,  0.0677,  0.0941, -0.1851, -0.1971, -0.0252, -0.0740,
         -0.0707,  0.0849],
        [-0.0332, -0.0078,  0.0635,  0.0289, -0.1846, -0.2806,  0.1143, -0.0762,
         -0.1332,  0.0184]], grad_fn=<AddmmBackward0>)

In [2]:
#自定义块
class MLP(nn.Module):
    def __init__(self):
        super().__init__()#初始化父类
        self.hidden=nn.Linear(20,256)# 隐藏层
        self.out=nn.Linear(256,10)# 输出层
    def forward(self,X):#定义前向传播逻辑
        return self.out(F.relu(self.hidden(X)))
net = MLP()
net(X)

tensor([[-0.1705, -0.0903,  0.0519,  0.3033,  0.0217,  0.0457, -0.1218,  0.0728,
         -0.1581,  0.2091],
        [-0.0583, -0.0918, -0.1292,  0.1978,  0.0154,  0.1158, -0.1737, -0.0359,
         -0.1236,  0.1428]], grad_fn=<AddmmBackward0>)

In [5]:
#顺序块
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for idx,module in enumerate(args):#enumerate()作用：这是 Python 的内置函数，用于遍历一个可迭代对象（如列表、元组），并同时返回当前元素的索引（计数）和元素本身。
            self._modules[str(idx)] = module
    def forward(self,X):
        for block in self._modules.values():
            X = block(X)
        return X
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.1419, -0.2217, -0.1122,  0.1482,  0.1271,  0.2578, -0.0542,  0.1927,
         -0.0093, -0.1522],
        [-0.1064, -0.0995,  0.0058,  0.2142,  0.1510,  0.2714,  0.1368,  0.0612,
         -0.0145, -0.1124]], grad_fn=<AddmmBackward0>)

In [6]:
#在前向传播函数中执行代码
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight=torch.rand((20,20),requires_grad=False)#权重（self.rand_weight）在实例化时被随机初始化，之后为常量
        self.linear=nn.Linear(20,20)
    def forward(self,X):
        X=self.linear(X)
        X=F.relu(torch.mm(X,self.rand_weight)+1)#先经过可训练线性层- 再和一个固定随机矩阵相乘- 再经过 `ReLU`
        X=self.linear(X)#两次用的是同一组参数，不是两套不同参数
        while X.abs().sum()>1:
            X/=2
        return X.sum()
net = FixedHiddenMLP()
net(X)

tensor(0.2038, grad_fn=<SumBackward0>)

In [7]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)
    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.1930, grad_fn=<SumBackward0>)